# TerminalSight — train Faster R-CNN and SSD-VGG16

Run each cell from top to bottom using its ▶ button. This notebook trains **Faster
R-CNN with ResNet-50-FPN** and **SSD300 with VGG16**, using the same Roboflow COCO
dataset. If your study requires another Faster R-CNN backbone, change the architecture
before training; checkpoints are architecture-specific.

**Upload:** the entire **COCO JSON dataset ZIP** from Roboflow, containing `train/`,
`valid/`, and preferably `test/`, their images, and `_annotations.coco.json` files.
Do not upload only a JSON file, a YOLO dataset ZIP, `best.pt`, or TerminalSight's codebase.
Both Bus and UV must have labeled examples in each split. Nested ZIP folders are supported.

**Before cell 1:** Runtime → Change runtime type → select a GPU (T4 if available) → Save.
Training uses Colab's NVIDIA GPU, not your computer's AMD card. GPU availability and
session duration are not guaranteed. Internet is needed for setup/pretrained weights;
the resulting files can later be used by the offline TerminalSight application.

Use the same dataset version/splits as YOLO for comparison. Keep related frames from
one recording in one split; this notebook only checks exact duplicates, not near-duplicates.
20 epochs is a starting experiment, not a promised accuracy. Validation chooses `best.pth`;
the test set is reserved for final evaluation.


## 1. Install the evaluation package and check the GPU
Keep Colab's matching PyTorch/Torchvision installation.


In [ ]:
%pip -q install pycocotools
import torch, torchvision, sys
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, '| Torchvision:', torchvision.__version__)
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU, then rerun this cell.'
print('GPU:', torch.cuda.get_device_name(0))
from torchvision.ops import nms
nms(torch.tensor([[0., 0., 10., 10.]], device='cuda'), torch.tensor([.9], device='cuda'), .5)
print('GPU and detection operators are ready.')


## 2. Create the training program
Run this whole cell once. You do not need to edit the long code.


In [ ]:
%%writefile /content/train_colab_detectors.py
"""Train TerminalSight detectors from a Roboflow COCO export (local or Colab).

This module is embedded in the companion notebook; it is not imported by Flask.
"""
import argparse
from collections import Counter, defaultdict
import contextlib
import hashlib
import io
import json
import math
from pathlib import Path
import random
import shutil
import zipfile

import PIL
from PIL import Image
import torch
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision.transforms import functional as TF

CLASS_NAMES = {0: '__background__', 1: 'bus', 2: 'uv'}
CLASS_ALIASES = {'bus': 1, 'buses': 1, 'uv': 2, 'uv express': 2, 'uvexpress': 2}
MODEL_NAMES = ('fasterrcnn_resnet50_fpn', 'ssd300_vgg16')


def extract_dataset(archive, destination):
    destination = Path(destination).resolve()
    if destination.exists() and any(destination.iterdir()):
        raise ValueError('Use an empty extraction directory to avoid mixing dataset versions.')
    with zipfile.ZipFile(archive) as bundle:
        for item in bundle.infolist():
            target = (destination / item.filename.replace('\\', '/')).resolve()
            if not target.is_relative_to(destination) or (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('ZIP contains an unsafe path or symbolic link.')
        destination.mkdir(parents=True, exist_ok=True)
        bundle.extractall(destination)
    return destination


def find_splits(root):
    result = {}
    for path in Path(root).rglob('_annotations.coco.json'):
        name = {'val': 'valid', 'validation': 'valid'}.get(path.parent.name.lower(), path.parent.name.lower())
        if name in ('train', 'valid', 'test'):
            if name in result:
                raise ValueError(f'Multiple {name} splits found. Extract just one dataset version.')
            result[name] = path
    if not {'train', 'valid'} <= result.keys():
        raise ValueError('Expected train/valid folders with _annotations.coco.json. Upload the COCO JSON ZIP, including images.')
    return result


def canonical_label(name):
    normalized = ' '.join(str(name).lower().replace('_', ' ').replace('-', ' ').split())
    if normalized not in CLASS_ALIASES:
        raise ValueError(f'Unexpected annotated class {name!r}. Confirm it is Bus or UV before editing CLASS_ALIASES.')
    return CLASS_ALIASES[normalized]


class CocoVehicles(Dataset):
    def __init__(self, annotation_path, augment=False):
        self.path = Path(annotation_path)
        self.augment = augment
        self.data = json.loads(self.path.read_text(encoding='utf-8-sig'))
        self.images = self.data['images']
        if not self.images:
            raise ValueError(f'{self.path}: empty image split')
        ids = [image['id'] for image in self.images]
        if len(set(ids)) != len(ids):
            raise ValueError('Duplicate COCO image IDs')
        categories = {category['id']: category['name'] for category in self.data['categories']}
        self.category_to_label = {}
        self.label_to_category = {}
        self.annotations = defaultdict(list)
        self.counts = Counter()
        image_ids = set(ids)
        for annotation in self.data['annotations']:
            if annotation['image_id'] not in image_ids:
                raise ValueError('Annotation refers to a missing image ID')
            category = annotation['category_id']
            label = canonical_label(categories[category])
            if label in self.label_to_category and self.label_to_category[label] != category:
                raise ValueError('Two annotated categories map to one vehicle class. Merge those labels in Roboflow.')
            self.category_to_label[category] = label
            self.label_to_category[label] = category
            if annotation.get('iscrowd', 0):
                raise ValueError('Crowd/group boxes found. Use individual vehicle bounding boxes for this notebook.')
            self.annotations[annotation['image_id']].append(annotation)
            self.counts[CLASS_NAMES[label]] += 1
        if set(self.label_to_category) != {1, 2}:
            raise ValueError(f'{self.path.parent.name}: needs annotated Bus AND UV examples; found {dict(self.counts)}')
        self.paths = []
        for record in self.images:
            path = (self.path.parent / record['file_name']).resolve()
            if not path.is_relative_to(self.path.parent.resolve()) or not path.is_file():
                raise ValueError(f'Missing image or invalid file_name: {record["file_name"]}')
            with Image.open(path) as image:
                if image.size != (record['width'], record['height']):
                    raise ValueError(f'Image size and annotations disagree: {path.name}')
                image.verify()
            # Validate every box now, before starting an expensive training run.
            self._boxes(record)
            self.paths.append(path)

    def _boxes(self, record):
        boxes, labels = [], []
        for annotation in self.annotations[record['id']]:
            x, y, width, height = annotation['bbox']
            if not all(math.isfinite(v) for v in (x, y, width, height)) or width <= 0 or height <= 0:
                raise ValueError(f'Invalid bounding box in image {record["id"]}')
            x1, y1 = max(0, x), max(0, y)
            x2, y2 = min(record['width'], x + width), min(record['height'], y + height)
            if x2 <= x1 or y2 <= y1:
                raise ValueError(f'Bounding box outside image {record["id"]}')
            boxes.append([x1, y1, x2, y2])
            labels.append(self.category_to_label[annotation['category_id']])
        return torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4), torch.tensor(labels, dtype=torch.int64)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        record = self.images[index]
        with Image.open(self.paths[index]) as original:
            image = original.convert('RGB')
        boxes, labels = self._boxes(record)
        if self.augment and random.random() < .5:
            image = TF.hflip(image)
            boxes[:, [0, 2]] = image.width - boxes[:, [2, 0]]
        target = {'boxes': boxes, 'labels': labels, 'image_id': torch.tensor(record['id'], dtype=torch.int64)}
        # Models apply their own normalization/resizing; preserve original coordinates here.
        return TF.to_tensor(image), target


def inspect_dataset(root):
    splits = find_splits(root)
    fingerprints = {}
    for name, path in splits.items():
        dataset = CocoVehicles(path)
        print(f'{name}: {len(dataset)} images, boxes={dict(dataset.counts)}', flush=True)
        for image in dataset.paths:
            digest = hashlib.sha256(image.read_bytes()).hexdigest()
            if digest in fingerprints and fingerprints[digest] != name:
                raise ValueError(f'Identical image occurs in both {fingerprints[digest]} and {name}: {image.name}')
            fingerprints[digest] = name
    print('Class IDs for training: 0=background, 1=bus, 2=uv. COCO IDs are remapped by NAME.')
    if 'test' not in splits:
        print('No test split found. Create a held-out test split before final model comparison.')
    return splits


def build_model(name, pretrained=True):
    from torchvision.models.detection import (
        fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights,
        ssd300_vgg16, SSD300_VGG16_Weights,
    )
    if name == 'fasterrcnn_resnet50_fpn':
        from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
        model = fasterrcnn_resnet50_fpn(
            weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1 if pretrained else None,
            weights_backbone=None, min_size=512, max_size=768,
        )
        features = model.roi_heads.box_predictor.cls_score.in_features
        model.roi_heads.box_predictor = FastRCNNPredictor(features, len(CLASS_NAMES))
        # Torchvision otherwise uses ordinary BatchNorm when weights=None, which
        # would change the architecture on offline reload/resume of a COCO model.
        from torchvision.ops.misc import FrozenBatchNorm2d

        def freeze_norm(module):
            for child_name, child in list(module.named_children()):
                if isinstance(child, torch.nn.BatchNorm2d):
                    frozen = FrozenBatchNorm2d(child.num_features, eps=child.eps)
                    frozen.load_state_dict(child.state_dict())
                    setattr(module, child_name, frozen)
                else:
                    freeze_norm(child)
        freeze_norm(model.backbone)
        for module in model.modules():
            if isinstance(module, FrozenBatchNorm2d):
                module.eps = 0.0  # Matches Torchvision's COCO_V1 checkpoint on offline reload.
    elif name == 'ssd300_vgg16':
        from torchvision.models.detection.ssd import SSDClassificationHead
        model = ssd300_vgg16(weights=SSD300_VGG16_Weights.COCO_V1 if pretrained else None, weights_backbone=None)
        channels = [layer.in_channels for layer in model.head.classification_head.module_list]
        anchors = model.anchor_generator.num_anchors_per_location()
        model.head.classification_head = SSDClassificationHead(channels, anchors, len(CLASS_NAMES))
    else:
        raise ValueError(f'Unknown model: {name}')
    # Same fine-tuning policy on initial training and checkpoint resume.
    for parameter in model.parameters():
        parameter.requires_grad_(True)
    return model


def collate(batch):
    return tuple(zip(*batch))


def loader(dataset, batch_size, shuffle=False):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0,
                      pin_memory=torch.cuda.is_available(), collate_fn=collate)


@torch.inference_mode()
def evaluate(model, dataset, device, batch_size=2):
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    model.eval()
    detections = []
    for images, targets in loader(dataset, batch_size):
        predictions = model([image.to(device) for image in images])
        for prediction, target in zip(predictions, targets):
            for box, label, score in zip(prediction['boxes'].cpu().tolist(), prediction['labels'].cpu().tolist(),
                                         prediction['scores'].cpu().tolist()):
                if label not in dataset.label_to_category:
                    continue
                x1, y1, x2, y2 = box
                detections.append({'image_id': int(target['image_id']),
                                   'category_id': dataset.label_to_category[label],
                                   'bbox': [x1, y1, x2-x1, y2-y1], 'score': score})
    if not detections:
        return {'map_50_95': 0.0, 'map_50': 0.0}
    with contextlib.redirect_stdout(io.StringIO()):
        truth = COCO(str(dataset.path))
        truth.dataset.setdefault('info', {})
        # Some Roboflow exports omit these optional COCO fields.
        for annotation in truth.dataset['annotations']:
            annotation.setdefault('area', annotation['bbox'][2] * annotation['bbox'][3])
            annotation.setdefault('iscrowd', 0)
        scorer = COCOeval(truth, truth.loadRes(detections), 'bbox')
        scorer.params.imgIds = [record['id'] for record in dataset.images]
        scorer.params.catIds = list(dataset.label_to_category.values())
        scorer.evaluate()
        scorer.accumulate()
        scorer.summarize()
    return {'map_50_95': float(scorer.stats[0]), 'map_50': float(scorer.stats[1])}


def save_checkpoint(payload, path):
    path = Path(path)
    temporary = path.with_suffix('.tmp')
    torch.save(payload, temporary)
    temporary.replace(path)


def checkpoint_metadata(name, train, seed):
    return {
        'format_version': 1, 'architecture': name, 'class_names': CLASS_NAMES,
        'category_to_label': train.category_to_label, 'seed': seed,
        'torch_version': str(torch.__version__), 'torchvision_version': str(torchvision.__version__),
        'pillow_version': str(PIL.__version__),
        'training_annotation_sha256': hashlib.sha256(train.path.read_bytes()).hexdigest(),
        'input': 'RGB float32 [0,1]; model performs internal normalization and resize',
        'resize': {'min_size': 512, 'max_size': 768} if name == MODEL_NAMES[0] else {'size': [300, 300]},
    }


def train_model(args):
    if not torch.cuda.is_available():
        raise RuntimeError('No CUDA GPU. In Colab: Runtime > Change runtime type > GPU, then rerun setup cells.')
    random.seed(args.seed)
    torch.manual_seed(args.seed)
    torch.cuda.manual_seed_all(args.seed)
    splits = find_splits(args.dataset)
    train = CocoVehicles(splits['train'], augment=True)
    valid = CocoVehicles(splits['valid'])
    output = Path(args.output)
    output.mkdir(parents=True, exist_ok=True)
    if (output / 'last.pth').exists() and not args.resume:
        raise ValueError('This folder already has a run. Use --resume with its last.pth or choose a new output folder.')
    device = torch.device('cuda')
    model = build_model(args.model, pretrained=not bool(args.resume)).to(device)
    optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                                lr=args.lr, momentum=.9, weight_decay=.0005)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=8, gamma=.2)
    metadata = checkpoint_metadata(args.model, train, args.seed)
    metadata.update(batch_size=args.batch_size, initial_lr=args.lr, planned_epochs=args.epochs)
    metadata['validation_annotation_sha256'] = hashlib.sha256(valid.path.read_bytes()).hexdigest()
    start, best_score, history = 0, -1.0, []
    if args.resume:
        previous = torch.load(args.resume, map_location='cpu', weights_only=True)
        for field in ('architecture', 'class_names', 'training_annotation_sha256',
                      'validation_annotation_sha256', 'initial_lr', 'seed'):
            if previous['metadata'][field] != metadata[field]:
                raise ValueError(f'Resume mismatch in {field}')
        if not (output / 'best.pth').is_file():
            raise ValueError('Resume into the same output folder containing best.pth and last.pth.')
        model.load_state_dict(previous['state_dict'])
        optimizer.load_state_dict(previous['optimizer'])
        scheduler.load_state_dict(previous['scheduler'])
        start, best_score, history = previous['epoch'], previous['best_map_50_95'], previous['history']
        random.setstate(previous['random_state'])
        torch.set_rng_state(previous['torch_rng'])
        torch.cuda.set_rng_state_all(previous['cuda_rng'])
    train_loader = loader(train, args.batch_size, shuffle=True)
    for epoch in range(start, args.epochs):
        model.train()
        total = 0.0
        for step, (images, targets) in enumerate(train_loader, 1):
            images = [image.to(device) for image in images]
            targets = [{key: value.to(device) for key, value in target.items()} for target in targets]
            loss = sum(model(images, targets).values())
            if not torch.isfinite(loss):
                raise RuntimeError('Non-finite training loss. Check annotations and lower the learning rate.')
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 10)
            optimizer.step()
            total += float(loss.detach())
            if step % 25 == 0:
                print(f'Epoch {epoch+1}/{args.epochs} batch {step}/{len(train_loader)} loss={total/step:.4f}', flush=True)
        metrics = evaluate(model, valid, device, args.batch_size)
        scheduler.step()
        history.append({'epoch': epoch+1, 'train_loss': total/len(train_loader), **metrics})
        print(json.dumps(history[-1]), flush=True)
        payload = {'metadata': metadata, 'epoch': epoch+1, 'state_dict': model.state_dict(), 'validation': metrics}
        if metrics['map_50_95'] > best_score:
            best_score = metrics['map_50_95']
            save_checkpoint(payload, output / 'best.pth')
            print('Saved new best validation checkpoint.', flush=True)
        save_checkpoint({**payload, 'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
                         'best_map_50_95': best_score, 'history': history,
                         'random_state': random.getstate(), 'torch_rng': torch.get_rng_state(),
                         'cuda_rng': torch.cuda.get_rng_state_all()}, output / 'last.pth')
        (output / 'metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
        (output / 'history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')
        if Path(__file__).resolve() != (output / 'train_colab_detectors.py').resolve():
            shutil.copyfile(__file__, output / 'train_colab_detectors.py')
    print(f'Finished. Best validation mAP@0.50:0.95: {best_score:.4f}; files: {output}', flush=True)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--dataset', required=True)
    parser.add_argument('--inspect', action='store_true')
    parser.add_argument('--model', choices=MODEL_NAMES)
    parser.add_argument('--output')
    parser.add_argument('--epochs', type=int, default=20)
    parser.add_argument('--batch-size', type=int, default=2)
    parser.add_argument('--lr', type=float, default=.002)
    parser.add_argument('--seed', type=int, default=42)
    parser.add_argument('--resume')
    parser.add_argument('--evaluate', help='best.pth to evaluate on the held-out test split')
    args = parser.parse_args()
    if args.inspect:
        inspect_dataset(args.dataset)
    elif args.evaluate:
        splits = find_splits(args.dataset)
        if 'test' not in splits:
            raise ValueError('No test split: do not substitute validation for held-out testing.')
        checkpoint = torch.load(args.evaluate, map_location='cpu', weights_only=True)
        model = build_model(checkpoint['metadata']['architecture'], pretrained=False)
        model.load_state_dict(checkpoint['state_dict'])
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        scores = evaluate(model.to(device), CocoVehicles(splits['test']), device, args.batch_size)
        path = Path(args.evaluate).parent / 'test_metrics.json'
        path.write_text(json.dumps(scores, indent=2), encoding='utf-8')
        print('Held-out TEST:', scores, flush=True)
    else:
        if not args.model or not args.output or args.epochs < 1 or args.batch_size < 1 or args.lr <= 0:
            parser.error('Training requires --model, --output, and positive epochs/batch-size/lr.')
        train_model(args)


if __name__ == '__main__':
    main()


## 3. Connect Google Drive to keep checkpoints
Authorize your own Drive in Colab's dialog. Each new notebook session uses a new run
folder. To resume after disconnecting, set `RUN_NAME` below to your previous folder
name and rerun setup/upload/validation cells. Then change the relevant training cell
to `resume=True`. Do not change the dataset or architecture when resuming.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime
import subprocess
RUN_NAME = 'run_' + datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_ROOT = Path('/content/drive/MyDrive/TerminalSight/training') / RUN_NAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
with (OUTPUT_ROOT / 'training_environment.txt').open('w') as target:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=target, check=True)
print('SAVE THIS RUN NAME for resuming:', RUN_NAME)
print('Checkpoint folder:', OUTPUT_ROOT)


## 4. Upload the COCO dataset ZIP
Click **Choose Files** when prompted and select the complete Roboflow ZIP from your
computer. For a large ZIP, upload it to Google Drive first, set `DRIVE_ZIP` to its
`/content/drive/MyDrive/...zip` path, and skip the browser upload. No Roboflow API key
is needed. Upload the dataset once; both models use it.


In [ ]:
from google.colab import files
DRIVE_ZIP = ''  # Optional: '/content/drive/MyDrive/terminalsight-coco.zip'
if DRIVE_ZIP:
    ZIP_PATH = Path(DRIVE_ZIP)
else:
    uploaded = files.upload()
    if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith('.zip'):
        raise ValueError('Choose exactly one COCO dataset ZIP.')
    ZIP_PATH = Path(next(iter(uploaded))).resolve()
    del uploaded
assert ZIP_PATH.is_file(), f'ZIP not found: {ZIP_PATH}'
print('Dataset ZIP:', ZIP_PATH)


## 5. Extract and validate the dataset
Wait for this cell to finish. It checks images, boxes, class names, and exact duplicate
images across splits. If it reports an unexpected class, fix/confirm your annotation
labels before training. COCO category ID 0 is supported: the loader remaps by class name.

If it reports **Identical image occurs in both...**, set `REPAIR_EXACT_DUPLICATES = True`
at the top of this cell and rerun it. This creates a separate cleaned copy, retaining
test copies first, then validation, then training. It removes matching annotation
records with each duplicate image. Original ZIP/files are untouched. Conflicting
annotations stop the repair for manual review by default. To exclude every copy of
these ambiguous images, also set `EXCLUDE_CONFLICTING_IMAGES = True`. This may shrink
any split, including test; loss of a class or an empty split still stops the repair. A removal report
and cleaned ZIP are saved to your run folder in Drive. Use that same cleaned dataset
for both models and any new YOLO comparison; previous scores are not directly comparable.


In [ ]:
REPAIR_EXACT_DUPLICATES = False
EXCLUDE_CONFLICTING_IMAGES = False

"""Make a separate COCO copy without byte-identical images across splits.

Retain test before validation before training. Original files are never modified.
Only exact file duplicates are detected, not related frames or augmented copies.
"""
from collections import defaultdict
import hashlib
import json
from pathlib import Path
import shutil


def _class_name(name):
    name = ''.join(str(name).lower().replace('_', ' ').replace('-', ' ').split())
    return {'buses': 'bus', 'uvexpress': 'uv'}.get(name, name)


def _signature(data, annotations):
    categories = {category['id']: _class_name(category['name']) for category in data['categories']}
    return sorted((categories[item['category_id']], tuple(float(v) for v in item['bbox']),
                   int(item.get('iscrowd', 0))) for item in annotations)


def repair_dataset(source, destination, *, conflict_policy='error'):
    """Optionally exclude every copy of ambiguously annotated images.

    The default stops for review. 'exclude' never chooses an annotation as correct;
    it drops the entire conflicting file-hash group, including held-out copies.
    """
    if conflict_policy not in ('error', 'exclude'):
        raise ValueError("conflict_policy must be 'error' or 'exclude'.")
    source, destination = Path(source).resolve(), Path(destination).resolve()
    if destination == source or destination.is_relative_to(source) or source.is_relative_to(destination):
        raise ValueError('The cleaned dataset must use a separate folder outside the original dataset.')
    if destination.exists() and any(destination.iterdir()):
        raise ValueError('Choose an empty output folder; existing files will not be overwritten.')
    splits = {}
    for path in sorted(source.rglob('_annotations.coco.json')):
        name = {'val': 'valid', 'validation': 'valid'}.get(path.parent.name.lower(), path.parent.name.lower())
        if name in ('test', 'valid', 'train'):
            if name in splits:
                raise ValueError(f'Multiple {name} splits found; use just one exported dataset version.')
            splits[name] = path
    if not {'train', 'valid'} <= splits.keys():
        raise ValueError('Expected train and valid COCO annotation files.')

    groups, originals = defaultdict(list), {}
    plans, removals, conflicts, failures = {}, [], [], []
    for name in ('test', 'valid', 'train'):
        if name not in splits:
            continue
        path = splits[name]
        data = json.loads(path.read_text(encoding='utf-8-sig'))
        by_image = defaultdict(list)
        for annotation in data['annotations']:
            by_image[annotation['image_id']].append(annotation)
        originals[name] = data
        for record in data['images']:
            image_path = (path.parent / record['file_name']).resolve()
            if not image_path.is_relative_to(path.parent.resolve()) or not image_path.is_file():
                raise ValueError(f'Missing image or unsafe image path: {record["file_name"]}')
            digest = hashlib.sha256(image_path.read_bytes()).hexdigest()
            signature = _signature(data, by_image[record['id']])
            groups[digest].append({'split': name, 'record': record, 'path': image_path,
                                   'signature': signature})

    kept_by_split = defaultdict(list)
    excluded_groups = 0
    for digest, entries in groups.items():
        first = entries[0]
        members = [{'split': entry['split'], 'image_id': entry['record']['id'],
                    'file_name': entry['record']['file_name'], 'signature': entry['signature']}
                   for entry in entries]
        ambiguous = any(entry['signature'] != first['signature'] for entry in entries[1:])
        if ambiguous:
            conflicts.append({'sha256': digest, 'members': members})
        if ambiguous and conflict_policy == 'exclude':
            excluded_groups += 1
            removals.extend({**member, 'sha256': digest, 'reason': 'annotation_conflict'}
                            for member in members)
            continue
        for entry, member in zip(entries, members):
            if entry['split'] != first['split']:
                removals.append({**member, 'sha256': digest, 'reason': 'cross_split_duplicate',
                                 'kept_split': first['split'], 'kept_file': first['record']['file_name']})
            else:
                kept_by_split[entry['split']].append(entry)

    for name, data in originals.items():
        kept = [entry['record'] for entry in kept_by_split[name]]
        kept_paths = [entry['path'] for entry in kept_by_split[name]]
        keep_ids = {record['id'] for record in kept}
        annotations = [item for item in data['annotations'] if item['image_id'] in keep_ids]
        before_classes = {item['category_id'] for item in data['annotations']}
        after_classes = {item['category_id'] for item in annotations}
        if not kept or before_classes != after_classes:
            failures.append(f'{name}: removing duplicates would empty the split or remove all examples of a class.')
        plans[name] = {'data': {**data, 'images': kept, 'annotations': annotations}, 'paths': kept_paths,
                       'original_images': len(data['images']), 'original_annotations': len(data['annotations'])}

    blocked = bool(failures or (conflicts and conflict_policy == 'error'))
    report = {'policy': 'Keep test, then valid, then train; optionally exclude all copies with conflicting annotations.',
              'conflict_policy': conflict_policy, 'excluded_conflicting_groups': excluded_groups,
              'source': str(source), 'removed': removals, 'annotation_conflicts': conflicts,
              'failures': failures, 'status': 'blocked' if blocked else 'complete',
              'splits': {name: {'before_images': plan['original_images'], 'after_images': len(plan['data']['images']),
                                'before_annotations': plan['original_annotations'],
                                'after_annotations': len(plan['data']['annotations'])} for name, plan in plans.items()}}
    destination.mkdir(parents=True, exist_ok=True)
    report_path = destination / 'deduplication_report.json'
    report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
    if blocked:
        raise ValueError(f'Repair stopped: {len(conflicts)} conflicting annotation group(s); {len(failures)} split-coverage issue(s). '
                         f'Review {report_path}. No cleaned image splits were written.')

    for name, plan in plans.items():
        folder = destination / name
        folder.mkdir()
        for record, image_path in zip(plan['data']['images'], plan['paths']):
            target = folder / record['file_name']
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(image_path, target)
        (folder / '_annotations.coco.json').write_text(json.dumps(plan['data'], indent=2), encoding='utf-8')
        print(f'{name}: kept {len(plan["data"]["images"])}/{plan["original_images"]} images')
    print(f'Excluded {excluded_groups} conflicting image group(s) from every split.')
    print(f'Removed {len(removals)} image records from the cleaned copy. Report: {report_path}')
    return destination


import tempfile
from train_colab_detectors import extract_dataset, inspect_dataset
DATA_ROOT = extract_dataset(ZIP_PATH, Path(tempfile.mkdtemp(prefix='terminalsight_coco_')))
if REPAIR_EXACT_DUPLICATES:
    cleaned = repair_dataset(DATA_ROOT, Path(tempfile.mkdtemp(prefix='terminalsight_clean_')),
                             conflict_policy='exclude' if EXCLUDE_CONFLICTING_IMAGES else 'error')
    inspect_dataset(cleaned)
    DATA_ROOT = cleaned
    shutil.copy2(cleaned / 'deduplication_report.json', OUTPUT_ROOT / 'deduplication_report.json')
    print('Saving the cleaned dataset ZIP to Drive for reuse/resume...')
    shutil.make_archive(str(OUTPUT_ROOT / 'cleaned_dataset'), 'zip', cleaned)
SPLITS = inspect_dataset(DATA_ROOT)
print('Data ready:', DATA_ROOT)


## 6. Check the labels visually
Confirm the boxes surround the correct buses and vans before proceeding.


In [ ]:
import matplotlib.pyplot as plt
from PIL import ImageDraw
from torchvision.transforms.functional import to_pil_image
from train_colab_detectors import CocoVehicles, CLASS_NAMES
preview = CocoVehicles(SPLITS['train'])
indices = [i for i, record in enumerate(preview.images) if preview.annotations[record['id']]][:3]
fig, axes = plt.subplots(1, len(indices), figsize=(6 * len(indices), 5), squeeze=False)
for axis, index in zip(axes[0], indices):
    tensor, target = preview[index]
    image = to_pil_image(tensor)
    draw = ImageDraw.Draw(image)
    for box, label in zip(target['boxes'].tolist(), target['labels'].tolist()):
        draw.rectangle(box, outline='lime', width=3)
        draw.text((box[0], box[1]), CLASS_NAMES[label], fill='red', stroke_width=1)
    axis.imshow(image); axis.axis('off')
plt.show()


## 7. Set training options
Start with batch size **2**. If CUDA runs out of memory, use **1** and rerun the relevant
training cell with `resume=True` if a completed epoch was saved. A disconnect during
an epoch resumes from the previous completed epoch. No completed epoch means start
that model again with `resume=False`. Epochs means the total target, including resumed epochs.


In [ ]:
EPOCHS = 20
BATCH_SIZE = 2
SEED = 42
def run_training(model_name, learning_rate, resume=False):
    output = OUTPUT_ROOT / model_name
    command = [sys.executable, '-u', '/content/train_colab_detectors.py',
               '--dataset', str(DATA_ROOT), '--model', model_name, '--output', str(output),
               '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
               '--lr', str(learning_rate), '--seed', str(SEED)]
    if resume:
        last = output / 'last.pth'
        assert last.is_file(), f'No completed epoch to resume: {last}'
        command += ['--resume', str(last)]
    subprocess.run(command, check=True)


## 8. Train Faster R-CNN (ResNet-50-FPN)
The first run downloads pretrained COCO weights. The classifier is replaced for
background/Bus/UV. Watch training loss and validation mAP; wait until **Finished**.


In [ ]:
run_training('fasterrcnn_resnet50_fpn', learning_rate=0.002, resume=False)


## 9. Train SSD300 (VGG16)
Run after Faster R-CNN finishes. It gets its own checkpoint folder.


In [ ]:
run_training('ssd300_vgg16', learning_rate=0.001, resume=False)


## 10. Review validation curves
`map_50` is mAP@0.50; `map_50_95` averages IoU 0.50 through 0.95. Values are fractions:
0.80 means 80%. The best checkpoint is selected by validation `map_50_95`, not training
loss and not test results. The two metrics are not interchangeable with your YOLO figure.


In [ ]:
import json
MODELS = ('fasterrcnn_resnet50_fpn', 'ssd300_vgg16')
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for model_name in MODELS:
    history = json.loads((OUTPUT_ROOT / model_name / 'history.json').read_text())
    for axis, key in zip(axes, ('train_loss', 'map_50', 'map_50_95')):
        axis.plot([row['epoch'] for row in history], [row[key] for row in history], label=model_name)
        axis.set_title(key); axis.set_xlabel('Epoch'); axis.legend()
plt.tight_layout(); plt.show()


## 11. Final held-out test evaluation
Run when your model/hyperparameter choices are final. Do not tune on these test results.
If your ZIP has no test split, this cell reports that final evaluation is missing; it
does not reuse validation as test data.


In [ ]:
if 'test' not in SPLITS:
    print('No held-out test split. Re-export with a separate test split for final evaluation.')
else:
    for model_name in MODELS:
        checkpoint = OUTPUT_ROOT / model_name / 'best.pth'
        subprocess.run([sys.executable, '-u', '/content/train_colab_detectors.py',
                        '--dataset', str(DATA_ROOT), '--evaluate', str(checkpoint),
                        '--batch-size', str(BATCH_SIZE)], check=True)
        print(model_name, (checkpoint.parent / 'test_metrics.json').read_text())


## 12. Download the model package
Downloads **TerminalSight_trained_models.zip** with both best checkpoints, class maps,
metadata, training source, history and any test metrics. The larger `last.pth` files
remain in Drive for resuming, and are intentionally excluded from the download.

Extract this ZIP locally and give Codex its path to integrate the models. These are
complete detection models, not only backbone weights. ONNX/OpenVINO exports and actual
AMD/Intel/NVIDIA runtime checks come next; this notebook does not claim those GPU paths
are already working. Never replace the existing YOLO `best.pt` with these checkpoints.


In [ ]:
import shutil
package = Path(tempfile.mkdtemp(prefix='terminalsight_models_'))
for model_name in MODELS:
    source = OUTPUT_ROOT / model_name
    destination = package / model_name
    destination.mkdir()
    for name in ('best.pth', 'metadata.json', 'history.json', 'train_colab_detectors.py'):
        if not (source / name).is_file():
            raise FileNotFoundError(f'Training output is incomplete: {source / name}')
        shutil.copy2(source / name, destination / name)
    if (source / 'test_metrics.json').is_file():
        shutil.copy2(source / 'test_metrics.json', destination / 'test_metrics.json')
    (destination / 'class_names.json').write_text(json.dumps(CLASS_NAMES, indent=2))
shutil.copy2(OUTPUT_ROOT / 'training_environment.txt', package / 'training_environment.txt')
archive = Path(shutil.make_archive('/content/TerminalSight_trained_models', 'zip', package))
shutil.copy2(archive, OUTPUT_ROOT / archive.name)
print('Backup in Google Drive:', OUTPUT_ROOT / archive.name)
files.download(str(archive))


## References and local validation
- [Torchvision detection fine-tuning](https://docs.pytorch.org/tutorials/intermediate/torchvision_tutorial.html)
- [Faster R-CNN](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.detection.fasterrcnn_resnet50_fpn.html)
- [SSD300-VGG16](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.detection.ssd300_vgg16.html)
- [Colab FAQ](https://research.google.com/colaboratory/faq.html)

The notebook has no saved training results. Local checks cover dataset handling,
notebook syntax and model construction/inference. Your actual dataset training and
accuracy must be validated in your Colab GPU session.
